# Anchor + Cousin Stress-Test  (16 known-hard classes)

Hammer the **8 named anchors** (AK(3)-AK(8) + Length-14) and **8 cousins** (`short_hard`,
`total_len<=13`) with the project's strongest available search, to check whether their
high/censored labels survive serious compute **before** the training archive trusts them.

- **Greedy** (numba GS-Sub, env-consistent per-relator <=24, RAM-bounded) + the AK-equivalence
  certificate scan (a cousin reaching an AK(k) class is *proven* AC-equivalent to it).
- **Beam** (610model policy, B=32,768 x 5 seeds, horizon T=150) -- the binding escalation; every
  solve is replay-validated.

**Verdict language is deliberate:** *"survived the strongest available search at budget X"*, never
*"proven hard"* -- neither engine is a neutral hardness oracle (greedy is length-greedy GBFS; the
610model policy is OOD on these basins). A **named-anchor "solve" -> HALT/escalate** (bug or an
Andrews-Curtis refutation), never relabel; **only cousins** take the relabel track.

> **Note on the greedy budget.** The greedy *frontier* grows ~45 states per popped node, so RAM --
> not the node count -- is the binding limit (~406 B/state measured; 10M nodes ~= 470M states ~= 190 GB).
> `## 8` therefore RAM-bounds greedy via `max_seen`, sized from a pre-flight to the largest depth that
> fits Colab RAM (~1M nodes), which is exactly the GS-Sub plateau (PLAN.md:117 -- 640@1M == 640@10M).

> **Verification status.** Setup (`## 1-5`), the greedy solver/worker (`## 7-8`, serial), and the
> verdict logic (`## 13`) are dry-run-verified locally (CPU). **`[unverified]`: `## 6`, `## 9-12`,
> and `## 14(d)` are the GPU/Colab path and have NOT been executed yet** -- they are derived verbatim
> from the proven `2.beam_harvest_pilot.ipynb` and interface-checked, but run them first on a clean A100.

Reviewed at plan time by `advisor()` + the Field Advisor (`tmp/field_advisor_pre.md`).

## 1. XLA flags (must run FIRST, before any JAX import)

In [ ]:
# Cell must run FIRST: XLA flags only take effect before jax is imported.
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # grow memory on demand; required for
os.environ["XLA_FLAGS"] = "--xla_gpu_autotune_level=0"  # honest memory_stats and wave sizing
print("XLA configured: no preallocation, autotune off.")

## 2. Repo setup (clone `test/eda` + install CUDA deps; local = no-op)

In [ ]:
import subprocess, sys
from pathlib import Path
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH = "test/eda"
print("Environment:", "Google Colab" if IN_COLAB else "local")

if IN_COLAB:
    REPO_DIR = Path("/content/ACSolverX")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-cuda.txt"], check=True)
else:
    REPO_DIR = Path.cwd()
    while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / "requirements.txt").exists():
        REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)
    print("Local repo:", REPO_DIR, "(skipping clone/install)")

print("cwd:", os.getcwd())
print("branch:", subprocess.run(["git", "branch", "--show-current"],
                                 capture_output=True, text=True).stdout.strip())

## 3. Mount Google Drive (durable, resumable output)

In [ ]:
# Mount Google Drive for durable, resumable output (survives Colab preemption).
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = Path(os.getcwd()) / "_stress_local"   # local fallback

RUN_TAG = "anchor_cousin_stress"      # fresh Drive folder per run; change to start a clean run
RUN_DIR = DRIVE_ROOT / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

## 4. Stress configuration + resumable JSONL helpers

In [ ]:
# Stress configuration -- one dict drives the whole notebook.
CONFIG = {
    # --- inputs (the labels under test) ---
    "ANCHORS_JSONL": "data/derived/dot/anchors.jsonl",      # 8 named anchors (censored, bound_B=150)
    "TRAP_SET":      "data/derived/dot/ak_trap_set.json",   # 16 keys: 8 named + 8 short_hard cousins
    "DOT_ARCHIVE":   "data/derived/labels/dot_archive.jsonl",
    "MERGED_INDEX":  "data/derived/paths/merged_best_paths_index.csv",  # for positive controls
    # the 16-target + 2-control dataset this notebook writes (## 5) and the env loads (## 6):
    "TARGETS_FILE":  "data/stress_targets.txt",            # bare stem "stress_targets"
    "DATASET_STEM":  "stress_targets",

    # --- greedy (numba, CPU, env-consistent per-relator <= 24) ---
    "GREEDY_NODES":    10_000_000,   # per-target node budget (GS-Sub plateaus by 1M; 10M = headroom)
    "PER_RELATOR_CAP": 24,           # mirrors the env's silent no-op on relator length > 24
    "GREEDY_WORKERS":  None,         # None -> auto from the RSS pre-flight (hard cap 8)
    "GREEDY_WORKER_CAP": 8,
    "PREFLIGHT_NODES": 200_000,      # one worker at this budget -> measure bytes/state ratio -> size pool
    "CONTROL_GREEDY_NODES": 200_000, # quick positive-control solve budget

    # --- beam (610model policy, B=32,768 x 5 seeds) ---
    "B": 32768,
    "T": 250,
    "ALPHA": 0.0,
    "SEEDS": [0, 1, 2, 3, 4],        # seed 0 deterministic (temp 0); 1-4 Gumbel explore
    "TEMP": 1.0, "TEMP_END": 0.0,    # explore-seed linear schedule 1.0 -> 0.0
    "SUSTAINED_SEED": 4,             # FA B5: one explore seed holds a higher constant temp
    "SUSTAINED_TEMP": 1.5,
    "WAVE_CAP": 3,                   # HARD cap: peak ~ W*B; W=3*32768 == proven-OK W=6*16384 (~59 GB)
    "CKPT": "610model",
    "TRAINING_DATASET": "AC19_extended",
    "RUN_GUARD": True,               # ## 9: True first; then set False + Runtime->Restart for the heavy run

    # --- output (durable, resumable) ---
    "RUN_TAG": "anchor_cousin_stress",
    "RESULTS_NAME": "anchor_stress_results.jsonl",
    "SUMMARY_NAME": "anchor_stress_summary.json",
    "REPO_RESULTS": "data/derived/dot/anchor_stress_results.jsonl",  # repo mirror (committed)
    "REPO_SUMMARY": "data/derived/dot/anchor_stress_summary.json",
}
for k, v in CONFIG.items():
    print(f"{k:20s} {v}")

# --- durable, resumable JSONL (greedy ## 8 + beam ## 11 both append here; fsync per record) ---
import os, json
RESULTS_JSONL = RUN_DIR / CONFIG["RESULTS_NAME"]
SUMMARY_PATH  = RUN_DIR / CONFIG["SUMMARY_NAME"]

def _rec_id(rec):
    if rec.get("rec_type") == "greedy":
        return ("greedy", rec["key"])
    if rec.get("rec_type") == "beam":
        return ("beam", rec["key"], rec.get("seed"))
    return (rec.get("rec_type"), rec.get("key"), rec.get("seed"))

def load_records():
    recs = []
    if RESULTS_JSONL.exists():
        with open(RESULTS_JSONL) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    recs.append(json.loads(line))
                except json.JSONDecodeError:
                    continue                      # tolerate a half-written final line from a crash
    return recs

def load_done_ids():
    return {_rec_id(r) for r in load_records()}

def append_record(rec):
    with open(RESULTS_JSONL, "a") as f:
        f.write(json.dumps(rec) + "\n")
        f.flush(); os.fsync(f.fileno())

print("results JSONL:", RESULTS_JSONL)

## 5. Build the 16 targets (+ 2 positive controls); GATE against `ak_trap_set.json`

In [ ]:
# Build the 16 stress targets (8 named anchors + 8 cousins) + 2 positive controls.
# Writes a single dataset file (data/stress_targets.txt): lines 0..15 = targets, 16..17 = controls.
# The beam env (## 6) loads this by index; greedy (## 8) reads the (r1, r2) strings directly.
import os, sys, json, csv, ast
sys.path.insert(0, "scripts")          # canon lives under scripts/lib via scripts/canon shim
sys.path.insert(0, "scripts/lib")
import canon

# --- 8 named anchors (already canonical in anchors.jsonl) ---
named = []
with open(CONFIG["ANCHORS_JSONL"]) as f:
    for line in f:
        d = json.loads(line)
        named.append({"r1": d["r1"], "r2": d["r2"], "group": "named",
                      "name": d.get("group", "named"), "total_len": d["total_len"]})
# recover the human label (AK(k)/Length-14) from the trap-set named map (key -> label)
trap = json.load(open(CONFIG["TRAP_SET"]))
key2label = {v: k for k, v in trap["groups"]["named"].items()}   # "r1|r2" -> "AK(3)"...
for t in named:
    t["key"] = canon.canon_key(t["r1"], t["r2"])[0]
    t["name"] = key2label.get(t["key"], t["name"])

# --- 8 cousins (short_hard keys; canonical "r1|r2" already) ---
cousins = []
for k in trap["groups"]["short_hard"]:
    r1, r2 = k.split("|")
    cousins.append({"r1": r1, "r2": r2, "group": "short_hard",
                    "name": f"cousin[{k}]", "total_len": len(r1) + len(r2),
                    "key": canon.canon_key(r1, r2)[0]})

TARGETS = named + cousins                       # 16, order: named then cousins
assert len(TARGETS) == 16, len(TARGETS)

# --- GATE: the 16 target keys must equal the labelled trap-set keys ---
target_keys = {t["key"] for t in TARGETS}
trap_keys = set(trap["keys"])
assert target_keys == trap_keys, (
    f"target/trap key mismatch: only-in-targets={target_keys - trap_keys}, "
    f"only-in-trap={trap_keys - target_keys}")
print(f"GATE OK: 16 target keys == ak_trap_set keys ({len(trap_keys)}).")

# AK certificate key-set (the 8 named, for the cousin certificate scan in ## 8)
AK_KEYS = {(t["r1"], t["r2"]): t["name"] for t in named}   # canonical (r1,r2) -> label

# --- 2 positive controls: short, greedy AND beam solvable (## 9/## 10 sanity) ---
CONTROLS = []
with open(CONFIG["MERGED_INDEX"]) as f:
    for row in csv.DictReader(f):
        g = str(row["greedy_solved"]).lower() in ("true", "1")
        b = str(row["beam_solved"]).lower() in ("true", "1")
        try:
            bl = int(row["beam_path_length"])
        except Exception:
            bl = -1
        r1, r2 = row["r1"], row["r2"]
        if g and b and 0 < bl <= 10 and len(r1) + len(r2) <= 12:
            CONTROLS.append({"r1": r1, "r2": r2, "group": "control",
                             "name": f"control[idx{row['idx']}]",
                             "key": canon.canon_key(r1, r2)[0], "beam_len": bl})
        if len(CONTROLS) == 2:
            break
assert len(CONTROLS) == 2, f"need 2 positive controls, got {len(CONTROLS)}"
print("controls:", [(c["name"], c["r1"], c["r2"], c["beam_len"]) for c in CONTROLS])

# --- write the dataset file (targets then controls) ---
ROWS = TARGETS + CONTROLS                        # 18 rows
CONTROL_IDX = [16, 17]
os.makedirs(os.path.dirname(CONFIG["TARGETS_FILE"]) or ".", exist_ok=True)
with open(CONFIG["TARGETS_FILE"], "w") as f:
    for r in ROWS:
        lit = canon.strs_to_presentation_literal(r["r1"], r["r2"], max_length=24)
        f.write(str(lit) + "\n")

# round-trip check: re-read line i -> same canonical key
with open(CONFIG["TARGETS_FILE"]) as f:
    for i, line in enumerate(f):
        x = ast.literal_eval(line.strip())
        rr1, rr2 = canon.env_state_to_strs(x, max_length=24)
        assert canon.canon_key(rr1, rr2)[0] == ROWS[i]["key"], f"round-trip fail line {i}"
print(f"wrote {CONFIG['TARGETS_FILE']}: {len(ROWS)} rows "
      f"(0-15 targets, {CONTROL_IDX} controls); round-trip OK.")
for i, t in enumerate(TARGETS):
    print(f"  [{i:2d}] {t['group']:10s} {t['name']:18s} len={t['total_len']:2d} {t['key']}")

## 6. Imports, beam env, 610model checkpoint, shared constants  *(`[unverified]` -- GPU/Colab path, not executed locally)*

In [ ]:
# Imports, beam env, 610model checkpoint, shared engine constants.
import time, math
import numpy as np
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
jax.config.update("jax_default_matmul_precision", "float32")

sys.path.insert(0, "scripts")                       # canon (also added in ## 5)
from envs.ac_s import ACS
from envs.utils import decode_action, replay_packed_path
from network import RelativeDualRingActorCritic
import canon

print("JAX devices:", jax.devices())
try:
    _bl = jax.devices()[0].memory_stats().get("bytes_limit", 0)
    print(f"device memory limit: {_bl/1e9:.1f} GB")
except Exception as e:
    print("memory_stats unavailable:", e)

B = CONFIG["B"]; T = CONFIG["T"]; L = 24; ALPHA = float(CONFIG["ALPHA"])
assert L == 24, "action-packing invariant: L must be 24 (packing formula + n_actions=2*2*L*L depend on it)"
assert isinstance(T, int) and T > 0, "search horizon T must be a positive int (a tunable knob, not an invariant)"

# Beam env loads the 18-row stress dataset (0-15 targets, 16-17 controls).
# max_steps_in_episode = the SEARCH horizon T (NOT the training NUM_STEPS=96) -- the documented
# [TRAP]: a smaller value would kill every beam at step 96 before a long path can finish.
env = ACS(n_gen=2, max_length=L, max_steps_in_episode=T, is_reward_sparse=False,
          initial_states_file=CONFIG["DATASET_STEM"])
env_params = env.default_params
network = RelativeDualRingActorCritic(activation="gelu")
n_actions = env.num_actions                         # 2304 = 2*2*24*24 (do NOT change)
obs_shape = env.observation_space(env_params).shape
obs_len = obs_shape[0]
print("n_actions:", n_actions, "obs_len:", obs_len)

# Restore 610model params only (search needs nothing else; solve_data/config skipped).
rng = jax.random.PRNGKey(0); rng, init_rng = jax.random.split(rng)
dummy_params = network.init(init_rng, jnp.zeros((1, *obs_shape)))
ckpt_abs = os.path.join(os.getcwd(), "ppo_checkpoints", CONFIG["CKPT"])
mngr = ocp.CheckpointManager(ckpt_abs, item_names=("params", "solve_data", "config"))
step = mngr.latest_step(); assert step is not None, f"no checkpoint at {ckpt_abs}"
restored = mngr.restore(step, args=ocp.args.Composite(params=ocp.args.StandardRestore(dummy_params)))
params = restored.params
print(f"restored {CONFIG['CKPT']} step {step} (params only).")

# Shared constants consumed by the batched beam engine (## 9), verbatim from beam_harvest ## 6.
GLOBAL_VISIT_CAP = B * T
HASH_SENTINEL = jnp.iinfo(jnp.int32).max
hash_vec = jax.random.randint(jax.random.PRNGKey(0xACABDED), (obs_len,),
                              minval=-(2**31), maxval=2**31 - 1, dtype=jnp.int32)

## 7. Greedy solver (self-contained; numba-optional; per-relator<=24; max_seen RAM guard)

In [ ]:
# === Greedy GS-Sub solver -- copied from greedy_search.ipynb cells 0-2, with three edits:
#   (1) numba-optional shim (mirrors scripts/lib/canon.py) so this runs in numba-less envs too;
#   (2) neighbor gate -> env-consistent PER-RELATOR <=24 cap (the env silently no-ops moves > 24);
#   (3) a max_seen RAM guard (the frontier grows ~tens of states/node; RAM, not nodes, is binding).
# Always run with stop_early=False (the AK certificate is a post-hoc new_seen scan in ## 8).
import heapq, time, random
import numpy as np
try:
    from numba import njit
except ImportError:                       # numba-optional passthrough
    def njit(*args, **kwargs):
        if len(args) == 1 and callable(args[0]) and not kwargs:
            return args[0]
        def deco(f):
            return f
        return deco

counterexamples = {('xyxYXY','x'*n+'Y'*(n+1)):f'AK({n})' for n in range(3,8)}
counterexamples[('XyyxYYY', 'XyxxyXX')] = 'Length 14 #1'
counterexamples[('XyyxYYY', 'XyxxYXX')] = 'Length 14 #2'

# Map characters to integers and back

# We store the group elements as pairs of booleans. This is helps make comparisons and inversions faster.
# The first boolean represents the generator (x or y) and the second boolean represents inversion (x or X).
char_to_array = {'x': [True, True], 'X': [True, False], 'y': [False, True], 'Y': [False, False]}

# We use a function instead of a dictionary for better performance with Numba.
# This function is called a lot.
@njit(inline='always')
def array_to_char(bool1, bool2):
    if bool1:
        if bool2:
            return 'x'
        return 'X'
    elif bool2:
        return 'y'
    return 'Y'

# -----------------------------------------------------
# Numba helpers with boolean arrays
# -----------------------------------------------------

@njit(inline='always')
def is_inverse_nj(a, b):
    '''
        Check if two group elements are inverses of each other.

        The elements are assumed to be in the form [x, y] where x, y are bools.
    '''
    return (a[0] == b[0]) and (a[1] != b[1])

@njit(inline='always')
def is_equal_nj(a, b):
    '''
        Check if two group elements are equal.

        The elements are assumed to be in the form [x, y] where x, y are bools.
    '''
    return (a[0] == b[0]) and (a[1] == b[1])

@njit(inline='always')
def is_less_than(a, b):
    '''
        Returns a < b with order defined as the lexicographic order with True > False.

        The elements are assumed to be in the form [x, y] where x, y are bools.
    '''
    if a[0] != b[0]:
        # If the first elements are not equal, then either a = True, b = False or a = False, b = True
        # In either case, a < b == b.
        return b[0]
    else:
        # If the first elements are equal, compare the second elements.
        return a[1] < b[1]

@njit
def inverse_relator_nj(rel=np.array([[]])):
    '''
        Invert a relator.
        
        We assume that rel is a relator in array form. (Numpy array of shape (n, 2))
        Thus inversion corresponds to reversing the order and flipping the boolean in the second coordinate.
    '''
    res = rel.copy()
    res = np.flipud(res)
    res[:, 1] = np.logical_not(res[:, 1])  # Flip the second element of each pair

    return res

@njit
def reduce_relator_nj(rel):
    ''' 
        Reduce a relator by removing joined pairs of inverses. Does not modify the input array.

        Assumes that rel is a relator in array form. (Numpy array of shape (n, 2))
        Also removes pairs of inverses cyclically.

        TODO: Is it fine to modify rel in place? If so, we can remove the copy operation below.
    '''

    n = len(rel)

    # rel_list will contain the reduced array.
    rel_list = np.zeros_like(rel)
    rel_list[0] = rel[0]

    # Current index tracks how many elements are initialized in rel_list.
    # It increases when we add a new element and decreases when we come across an inverse pair.
    current_index = 0

    # add_index is the index of the next element to be added from rel.
    add_index = 1

    # The goal is to minimize the number of passes through the array.
    while add_index < n: # TODO: Could refactor slightly to use a for loop here which might be faster in some cases?
        if is_inverse_nj(rel_list[current_index], rel[add_index]):
            # If the current and next elements are inverses, remove the current element.
            if current_index == 0:
                # If current_index is 0 and we have an inverse, our output rel would be empty
                # so we restart it with the next element.
                rel_list[0] = rel[add_index + 1]
                add_index += 2
            else:
                # If current_index is not 0, we remove the last element.
                add_index += 1
                current_index -= 1
        else:
            # If the current and next elements are not inverses, we add the next element to rel_list.
            current_index += 1
            rel_list[current_index] = rel[add_index]
            add_index += 1
                
    rel_list = rel_list[:current_index + 1]  # Current index is the last valid index.
    
    # Note that rel_list cannot be 0 for any trivial presentation.
    # If the first and last elements are inverses, we can remove them cyclically
    # and check to see if we can remove any more pairs.
    if is_inverse_nj(rel_list[0], rel_list[-1]):
        i = 1
        half_len = len(rel_list) / 2
        while i < half_len and is_inverse_nj(rel_list[i], rel_list[-1 - i]):
            i += 1
        rel_list = rel_list[i:-i]  # Remove the cyclic inverses.

    return rel_list

@njit
def find_minimal_rotation(rel):
    '''
        Find the minimal rotation of a string using Booth's algorithm.

        Uses the ordering `Y < y < X < x` for the characters.

        TODO: It's probably possible to speed this up using some more knowledge from our problem.
        E.g. We only really need to consider relator options which start with Y.
    '''

    n = len(rel)
    rel = np.concatenate((rel, rel))
    f = np.full(2 * n, -1, dtype=np.int32)
    k = 0
    for j in range(1, 2 * n):
        i = f[j - k - 1]
        while i != -1 and (not is_equal_nj(rel[j], rel[k + i + 1])):
            if is_less_than(rel[j], rel[k + i + 1]):
                k = j - i - 1
            i = f[i]
        if i == -1 and (not is_equal_nj(rel[j], rel[k])):
            if is_less_than(rel[j], rel[k]):
                k = j
            f[j - k] = -1
        else:
            f[j - k] = i + 1
    return rel[k:k + n]

@njit
def lex_cmp_array(a, b):
    """
        Compare two numpy arrays of shape (n, 2) with bool types in lexiographic order.
        Returns a >= b.
        
        Note that when comparing, True > False so the implied order on symbols is: Y < y < X < x.
    """
    for x, y in zip(a, b):
        if x[0] == ~y[0]:
            # If the elements are different then either x = True, y = False or x = False, y = True
            #  In either case, x > y == x.
            return x[0]
        elif x[1] == ~y[1]:
            return x[1]
    
    # If we reach here the arrays are equal.
    return True

@njit
def lex_cmp_pair(a, b):
    """
        Compare two 1D numpy arrays of length 2 with bool types in lexiographic order.
        Returns a > b.

        Note that when comparing, True > False so the implied order on symbols is: Y < y < X < x.
    """
    if a[0] == ~b[0]:
        # If the elements are different then either a = True, b = False or a = False, b = True
        #  In either case, a > b = a.
        return a[0]
    elif a[1] == ~b[1]:
        return a[1]
    
    # If we reach here the arrays are equal.
    return False

@njit
def canonical_relator_nj(r):
    ''' 
        Find the canonical relator for a given relator.
    '''
    r_min = find_minimal_rotation(r)
    inv_min = find_minimal_rotation(inverse_relator_nj(r))
    
    # If r_min > inv_min return inv_min, else return r_min.
    if lex_cmp_array(r_min, inv_min):
        return inv_min
    return r_min


@njit
def canonical_pair_nj(r1, r2):
    ''' 
        Find the canonical pair of relators for a given pair of relators.
    
        TODO: Might be worth adding in some other transformations:
        E.g. x <-> X or x <-> y.
    '''
    cr1 = canonical_relator_nj(r1)
    cr2 = canonical_relator_nj(r2)

    # sort pairs by length then lex order
    if len(cr1) > len(cr2) or (len(cr1) == len(cr2) and lex_cmp_array(cr1, cr2)):
        (cr1, cr2) = (cr2, cr1)
    
    return cr1, cr2

@njit
def get_neighbors_nj(r1, r2):
    '''
        Get all substitution neighbors of a pair of relators r1 and r2.

        Given two relators r1 and r2, let `i` be an index such that `r1[i] = (r2[i])^{-1}`.
        Then the neighour is defined by `neighbour = np.concatenate(r1[:i+1], r2, r2[i+1:])`
        and we have two neighbors of `(r1, r2)` namely `(neighbour, r2)` and `(r1, neighbour)`.

        We find all neighbors of both `(r1, r2)` and `(r1, r2^{-1})`.
    '''
    results = []
    candidates = [r2, inverse_relator_nj(r2)]
    for idx_c in range(2):
        c = candidates[idx_c]
        len_r1 = len(r1)
        len_c = len(c)
        for i in range(len_r1):
            rot1 = np.roll(r1, 2*i)
            for j in range(len_c):
                rot2 = np.roll(c, 2*j)
                if len(rot1) > 0 and len(rot2) > 0 and is_inverse_nj(rot1[-1], rot2[0]):
                    neighbour = np.concatenate((rot1, rot2))

                    # Replace r1
                    results.append((neighbour, r2))
                    # Replace r2
                    results.append((r1, neighbour))
    return results

# Helper to convert string to bool array and back
def str_to_arr(s):
    ''' 
        Convert a string in xXyY to a numpy array of shape (n, 2) with bool types.
    '''
    return np.array([char_to_array[c] for c in s], dtype=bool)

@njit(inline='always')
def arr_to_str(arr):
    ''' 
        Convert a numpy array of shape (n, 2) with bool types to a string in xXyY.
    '''
    # We use a function here as it lets us use Numba's njit decorator for performance.
    chars = [array_to_char(c[0], c[1]) for c in arr]
    return ''.join(chars)

@njit(inline='always')
def state_to_key(state):
    ''' 
        Convert tuple of arrays to tuple of strings for dict keys
    '''
    return (arr_to_str(state[0]), arr_to_str(state[1]))

def canonical_pair_str(r1, r2):
    r1_arr = str_to_arr(r1)
    r2_arr = str_to_arr(r2)

    (canon_r1_arr, canon_r2_arr) = canonical_pair_nj(reduce_relator_nj(r1_arr), reduce_relator_nj(r2_arr))
    
    return arr_to_str(canon_r1_arr), arr_to_str(canon_r2_arr)

# ----------------------------------------------
# Main solver class
# ----------------------------------------------

class ACRelatorSolver:
    def __init__(self, r1, r2, max_nodes=10000, max_len = 100, visited=None, verbose=True, stop_early=True, max_seen=None):
        self.max_nodes = max_nodes
        # max_seen: RAM guard. The frontier (new_seen/visited) grows ~tens of states per popped
        # node, so the node budget is NOT the binding limit -- RAM is. Stop once new_seen hits this cap.
        self.max_seen = max_seen if max_seen is not None else float('inf')

        # Verbose mode controls what information is printed.
        self.verbose = verbose

        # stop_early controls whether we stop the search when we find a known counterexample.
        self.stop_early = stop_early

        # The input visited allows us to continue a search but passing in a collection of already visited states.
        if visited is None:
            self.visited = dict()
        else:
            self.visited = visited

        # The new_seen set is used to track newly seen states in the current search.
        self.new_seen = set()

        # The maximum length of relators we are willing to consider.
        # In general the smaller this is, the faster our search will be.
        self.max_len = max_len
        
        # Depth keeps track of the path length.
        # Mostly here in case we want to easily investigate check out BFS or A^* search.
        self.current_depth = 0
        self.max_depth = 0

        # The maximum and minimum priority of the states we have seen so far.
        self.max_priority = len(r1) + len(r2)
        self.min_priority = len(r1) + len(r2)

        # Priority queue for the search.
        self.pq = []
        r1_arr = str_to_arr(r1)
        r2_arr = str_to_arr(r2)
        self.initial_state = canonical_pair_nj(reduce_relator_nj(r1_arr), reduce_relator_nj(r2_arr))
        self.counterexamples = {canonical_pair_str(k[0], k[1]): v for k, v in counterexamples.items()}

    def solve(self):
        heapq.heappush(self.pq, (len(self.initial_state[0]) + len(self.initial_state[1]), 0, state_to_key(self.initial_state)))
        self.visited[state_to_key(self.initial_state)] = None
        self.new_seen = set()
        self.new_seen.add(state_to_key(self.initial_state))
        nodes_visited = 0

        while self.pq and nodes_visited < self.max_nodes and len(self.new_seen) < self.max_seen:
            _, depth, key = heapq.heappop(self.pq)
            nodes_visited += 1
            r1, r2 = self._key_to_state(key)

            if self.verbose:
                # if depth > self.max_depth:
                #     print(f"First state of depth {depth}, priority {len(r1)+len(r2)} ({arr_to_str(r1)},{arr_to_str(r2)})")
                #     self.max_depth = depth

                if len(r1) + len(r2) > self.max_priority:
                    print(f"First state of priority {len(r1) + len(r2)}, depth: {depth}, values: {len(r1)}, {len(r2)} ({arr_to_str(r1)},{arr_to_str(r2)}), nodes: {nodes_visited}")
                    self.max_priority = len(r1) + len(r2)

                if len(r1) + len(r2) < self.min_priority:
                    print(f"First state of priority {len(r1) + len(r2)}, depth: {depth}, values: {len(r1)}, {len(r2)} ({arr_to_str(r1)},{arr_to_str(r2)}), nodes: {nodes_visited}")
                    self.min_priority = len(r1) + len(r2)

            if len(r1) == 1 and len(r2) == 1:
                path = []
                state_key = key
                while state_key is not None:
                    path.append(self._key_to_state(state_key))
                    state_key = self.visited[state_key]
                path.reverse()
                # print("Found trivial relators!")
                # print(f"r1 = {arr_to_str(r1)}, r2 = {arr_to_str(r2)}, depth = {depth}")
                return path, nodes_visited, self.new_seen
            elif self.stop_early and key in self.counterexamples:
                print(f"Found counterexample {self.counterexamples[key]} at depth {depth}, r1 = {arr_to_str(r1)}, r2 = {arr_to_str(r2)}, nodes = {nodes_visited}")
                path = []
                state_key = key
                while state_key is not None:
                    path.append(self._key_to_state(state_key))
                    state_key = self.visited[state_key]
                path.reverse()
                return path, nodes_visited, self.new_seen

            for nr1, nr2 in get_neighbors_nj(r1, r2):
                nr1r = reduce_relator_nj(nr1)
                nr2r = reduce_relator_nj(nr2)
                
                if len(nr1r) <= 24 and len(nr2r) <= 24:   # env-consistent per-relator cap (was: sum < max_len)
                    canon_r1, canon_r2 = canonical_pair_nj(nr1r, nr2r)

                    key_new = state_to_key((canon_r1, canon_r2))
                    if key_new not in self.visited:
                        self.visited[key_new] = key
                        self.new_seen.add(key_new)
                        priority = len(canon_r1) + len(canon_r2) # normal GS priority
                        # priority = len(canon_r1) + len(canon_r2) + abs(len(canon_r1) - len(canon_r2))  # a new test priority
                        # if nodes_visited < 100000:
                        #     priority = 100-len(canon_r1) - len(canon_r2)  # another new test priority
                        # else:
                        #     priority = len(canon_r1) + len(canon_r2)
                        heapq.heappush(self.pq, (priority, depth+1, key_new))
        
        # If we reach here, we have not found any trivial relators.
        # It can be interesting to see what the minimal relators found are.
        # This is particularly useful when the blocks checking len(r1) + len(r2) are commented out.
        if self.verbose:
            print("No trivial relators found.")
            min_pres = min(self.new_seen, key=lambda k: len(k[0]) + len(k[1]))
            min_pres_r1 = min(self.new_seen, key=lambda k: (len(k[0]), len(k[1])))
            print(f"Minimal element found: r1 = {min_pres[0]}, r2 = {min_pres[1]}, Size: ({len(min_pres[0])}, {len(min_pres[1])})")
            print(f"Minimal element found: r1 = {min_pres_r1[0]}, r2 = {min_pres_r1[1]}, Size: ({len(min_pres_r1[0])}, {len(min_pres_r1[1])})")

        return None, nodes_visited, self.new_seen

    def _key_to_state(self, key):
        return (str_to_arr(key[0]), str_to_arr(key[1]))

## 8. Greedy stress run (RAM-bounded, parallel, resumable; AK certificate scan)

In [ ]:
# Greedy stress run -- RAM-bounded (frontier, not nodes, is the limit), parallel, resumable.
import multiprocessing as mp, resource, platform

def _rss_gb():
    kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return kb / 1e9 if platform.system() == "Darwin" else kb / 1e6   # macOS=bytes, Linux=KB

def _total_ram_gb():
    try:
        return os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 1e9
    except Exception:
        return 80.0

def _greedy_one(task):
    """Run ONE target to (max_nodes, max_seen); return a small summary (never the huge seen set)."""
    key, name, group, r1, r2, max_nodes, max_seen = task
    t0 = time.time()
    s = ACRelatorSolver(r1, r2, max_nodes=max_nodes, stop_early=False, verbose=False, max_seen=max_seen)
    path, nodes, seen = s.solve()
    solved = path is not None
    glen = len(path) - 1 if solved else -1
    best = min(seen, key=lambda k: len(k[0]) + len(k[1]))
    best_len = len(best[0]) + len(best[1])
    self_key = tuple(canon.canon_key(r1, r2)[0].split("|"))
    touched = next((lab for kk, lab in AK_KEYS.items() if kk != self_key and kk in seen), None)
    stop = ("solved" if solved else
            "seen_cap" if len(seen) >= max_seen else
            "node_cap" if nodes >= max_nodes else "frontier_exhausted")
    return {"rec_type": "greedy", "key": key, "name": name, "group": group,
            "greedy_solved": solved, "greedy_len": glen, "nodes_visited": int(nodes),
            "n_seen": int(len(seen)), "best_len_reached": int(best_len),
            "ak_key_touched": touched, "stop_reason": stop,
            "wall_time": round(time.time() - t0, 1), "max_nodes": int(max_nodes),
            "max_seen": (None if max_seen == float("inf") else int(max_seen)), "per_relator_cap": 24}

# --- pre-flight: warm numba BEFORE forking, then measure bytes/state + frontier growth ---
ACRelatorSolver("xy", "XY", max_nodes=10, stop_early=False, verbose=False).solve()   # compile njit tree
_t = time.time()
_pf = ACRelatorSolver("YXYxyx", "YYYYxxx", max_nodes=CONFIG["PREFLIGHT_NODES"],
                      stop_early=False, verbose=False)
_, pf_nodes, pf_seen = _pf.solve()
bytes_per_state = max(1.0, _rss_gb() * 1e9 / max(len(pf_seen), 1))
states_per_node = len(pf_seen) / max(pf_nodes, 1)
print(f"pre-flight: {pf_nodes:,} nodes -> {len(pf_seen):,} seen ({states_per_node:.0f}/node), "
      f"~{bytes_per_state:.0f} B/state, {_rss_gb():.1f} GB peak ({time.time()-_t:.0f}s)")
del _pf, pf_seen; import gc; gc.collect()       # free the preflight frontier BEFORE forking workers

# --- size WORKERS + SEEN_CAP to RAM, preferring depth (reach the ~1M-node GS-Sub plateau) ---
TOTAL_RAM = _total_ram_gb(); RAM_BUDGET = 0.70 * TOTAL_RAM
PLATEAU_NODES = 1_000_000                                  # PLAN.md:117 -- greedy gains nothing past 1M
gb_for_plateau = PLATEAU_NODES * states_per_node * bytes_per_state / 1e9
ncpu = mp.cpu_count()
WORKERS = CONFIG["GREEDY_WORKERS"] or max(
    1, min(ncpu, CONFIG["GREEDY_WORKER_CAP"], int(RAM_BUDGET // max(gb_for_plateau, 1))))
SEEN_CAP = int((RAM_BUDGET / WORKERS) * 1e9 / bytes_per_state)               # fill each RAM slice
SEEN_CAP = min(SEEN_CAP, int(CONFIG["GREEDY_NODES"] * states_per_node))      # never exceed node ceiling
eff_nodes = int(SEEN_CAP / max(states_per_node, 1))
print(f"RAM {TOTAL_RAM:.0f} GB (budget {RAM_BUDGET:.0f}) | WORKERS={WORKERS} | "
      f"SEEN_CAP={SEEN_CAP/1e6:.0f}M states (~{eff_nodes/1e6:.2f}M nodes, "
      f"~{SEEN_CAP*bytes_per_state/1e9:.0f} GB/worker)")
if eff_nodes < CONFIG["GREEDY_NODES"]:
    need = CONFIG["GREEDY_NODES"] * states_per_node * bytes_per_state / 1e9
    print(f">>> NOTE: the literal {CONFIG['GREEDY_NODES']/1e6:.0f}M-node budget is RAM-infeasible "
          f"(frontier ~= {CONFIG['GREEDY_NODES']*states_per_node/1e6:.0f}M states ~= {need:.0f} GB). "
          f"Effective budget = ~{eff_nodes/1e6:.2f}M nodes (RAM-bound). GS-Sub plateaus at 1M nodes "
          f"(PLAN.md:117), so this still exerts full greedy power. <<<")

# --- run (resumable); maxtasksperchild=1 frees each worker's huge seen set between targets ---
done = load_done_ids()
GTASKS = [(t["key"], t["name"], t["group"], t["r1"], t["r2"], CONFIG["GREEDY_NODES"], SEEN_CAP)
          for t in TARGETS if ("greedy", t["key"]) not in done]
GTASKS += [(c["key"], c["name"], "control", c["r1"], c["r2"], CONFIG["CONTROL_GREEDY_NODES"], SEEN_CAP)
           for c in CONTROLS if ("greedy", c["key"]) not in done]
print(f"greedy todo: {len(GTASKS)} (done already: {sum(1 for x in done if x[0]=='greedy')})")

if GTASKS:
    ctx = mp.get_context("fork")          # fork: workers inherit compiled njit + ACRelatorSolver/AK_KEYS
    with ctx.Pool(processes=WORKERS, maxtasksperchild=1) as pool:
        for rec in pool.imap_unordered(_greedy_one, GTASKS):
            append_record(rec)
            print(f"  [{rec['group']:9s}] {rec['name']:18s} solved={rec['greedy_solved']} "
                  f"glen={rec['greedy_len']} best_len={rec['best_len_reached']} "
                  f"AK={rec['ak_key_touched']} stop={rec['stop_reason']} "
                  f"nodes={rec['nodes_visited']:,} ({rec['wall_time']}s)")

# --- positive control: greedy MUST solve the controls (else the harness is broken) ---
gmap = {r["key"]: r for r in load_records() if r["rec_type"] == "greedy"}
for c in CONTROLS:
    assert gmap[c["key"]]["greedy_solved"], f"GREEDY POSITIVE CONTROL FAILED: {c['name']} did not solve"
print("greedy positive controls solved OK.")

## 9. Batched beam engine (B=32,768; verbatim from beam_harvest)  *(`[unverified]` -- GPU/Colab path, not executed locally)*

In [ ]:
# Batched beam engine -- make_engine(B) copied VERBATIM from beam_harvest ## 7
# (vmap over a wave of (idx, seed, temp, temp_end) elements; per-element temp schedule; two
#  vmap-safety fixes). Closes over network/params/env/n_actions/hash_vec/HASH_SENTINEL/ALPHA/L/T/B.
def make_engine(B):
    """Return (run_wave, run_batch) for beam width B. Closes over network/params/env/etc."""
    GLOBAL_VISIT_CAP = B * T

    def beam_step_batched(states, alive, cum_log_prob, action_seqs, visited_sorted, temp_sched, rng, t):
        # === verbatim from beam/beam_search.py:beam_step, with (1) temp_sched arg and
        #     (2) two vmap-safety fixes marked below. Logic is otherwise unchanged. ===
        rng, noise_rng, step_rng = jax.random.split(rng, 3)
        pi, value = network.apply(params, states.x)
        log_probs = jax.nn.log_softmax(pi.logits, axis=-1)
        action_scores = log_probs + ALPHA * value[:, None]
        u = jax.random.uniform(noise_rng, action_scores.shape, minval=1e-6, maxval=1.0 - 1e-6)
        gumbel = -jnp.log(-jnp.log(u))
        action_scores = action_scores + temp_sched[t] * gumbel              # per-element temp
        action_scores = jnp.where(alive[:, None], action_scores, -jnp.inf)
        candidate_scores = cum_log_prob[:, None] + action_scores
        flat_scores = candidate_scores.reshape(-1)
        top_vals, top_idx = jax.lax.top_k(flat_scores, B)
        parent = top_idx // n_actions
        flat_action = top_idx % n_actions
        action_k1 = (flat_action // (4 * L)) + 1
        rem = flat_action % (4 * L)
        action_k2_tmp = rem // 4
        ij = rem % 4
        action_i = ij // 2
        action_j = ij % 2
        action_k2 = action_k2_tmp * (-1) ** action_j - action_j
        action_vec = jnp.stack([action_i, action_j, action_k1, action_k2], axis=-1)
        parent_states = jax.tree.map(lambda v: v[parent], states)
        step_rngs = jax.random.split(step_rng, B)
        _, new_states, _, dones, info = jax.vmap(
            env.step_env, in_axes=(0, 0, 0, None))(step_rngs, parent_states, action_vec, env_params)
        new_action_seqs = action_seqs[parent].at[:, t].set(flat_action)
        noop_mask = jnp.all(parent_states.x == new_states.x, axis=-1)
        parent_alive = alive[parent]
        terminated = info["terminated"] & parent_alive
        new_alive = parent_alive & (~dones) & (top_vals > -1e30) & (~noop_mask)
        new_cum_log_prob = jnp.where(new_alive, top_vals, -jnp.inf)
        hashes = jnp.sum(new_states.x.astype(jnp.int32) * hash_vec[None, :], axis=1)
        sort_order = jnp.lexsort(jnp.stack([-new_cum_log_prob, hashes]))
        sorted_hashes = hashes[sort_order]
        # --- vmap fix 1: build is_first from a batched slice (no jnp.array([True]) constant) ---
        prev_hashes = jnp.concatenate([sorted_hashes[:1], sorted_hashes[:-1]])
        is_first = (sorted_hashes != prev_hashes).at[0].set(True)
        # --- vmap fix 2: zeros_like(alive), not zeros(B), as the scatter target ---
        keep = jnp.zeros_like(alive).at[sort_order].set(is_first)
        new_alive = new_alive & keep
        new_cum_log_prob = jnp.where(new_alive, new_cum_log_prob, -jnp.inf)
        idx_in_vis = jnp.searchsorted(visited_sorted, hashes)
        idx_clamped = jnp.minimum(idx_in_vis, GLOBAL_VISIT_CAP - 1)
        already_visited = (visited_sorted[idx_clamped] == hashes)
        new_alive = new_alive & (~already_visited)
        new_cum_log_prob = jnp.where(new_alive, new_cum_log_prob, -jnp.inf)
        new_entries = jnp.where(new_alive, hashes, HASH_SENTINEL)
        merged = jnp.concatenate([visited_sorted, new_entries])
        new_visited_sorted = jnp.sort(merged)[:GLOBAL_VISIT_CAP]
        return (new_states, new_alive, new_cum_log_prob, new_action_seqs,
                terminated, new_visited_sorted, rng)

    step_vmapped = jax.jit(jax.vmap(
        beam_step_batched, in_axes=(0, 0, 0, 0, 0, 0, 0, None)))

    def _init_element(idx):
        key = jax.random.PRNGKey(0)
        _, init_state = env.reset_env(key, env_params, idx=jnp.int32(int(idx)),
                                      sample=jnp.bool_(False), probs=None)
        bstate = jax.tree.map(
            lambda v: jnp.broadcast_to(jnp.asarray(v)[None], (B,) + jnp.asarray(v).shape),
            init_state)
        alive0 = jnp.ones(B, dtype=jnp.bool_)
        clp0 = jnp.full((B,), -jnp.inf, dtype=jnp.float32).at[0].set(0.0)
        aseq0 = jnp.full((B, T), -1, dtype=jnp.int32)
        init_hash = jnp.sum(jnp.asarray(init_state.x, dtype=jnp.int32) * hash_vec)
        vis0 = jnp.full((GLOBAL_VISIT_CAP,), HASH_SENTINEL, dtype=jnp.int32).at[0].set(init_hash)
        vis0 = jnp.sort(vis0)
        return bstate, alive0, clp0, aseq0, vis0

    def run_wave(elements, t_cap=None):
        # elements: list of (idx, seed, temp, temp_end); length is the (constant) wave size.
        W = len(elements)
        sl_, al_, cl_, aq_, vs_, tp_, rg_ = [], [], [], [], [], [], []
        for (idx, seed, temp, temp_end) in elements:
            s, a, c, q, v = _init_element(idx)
            sl_.append(s); al_.append(a); cl_.append(c); aq_.append(q); vs_.append(v)
            tp_.append(jnp.linspace(float(temp), float(temp_end), T))
            rg_.append(jax.random.PRNGKey(int(seed)))
        states = jax.tree.map(lambda *xs: jnp.stack(xs), *sl_)
        alive = jnp.stack(al_); clp = jnp.stack(cl_); aseq = jnp.stack(aq_)
        vis = jnp.stack(vs_); temp_sched = jnp.stack(tp_); rngs = jnp.stack(rg_)

        solved = [False] * W; solved_len = [-1] * W; packed = [None] * W
        done = [False] * W
        stop = T if not t_cap else min(int(t_cap), T)   # horizon cap (timeout); see "## 11b"
        for t in range(stop):
            states, alive, clp, aseq, terminated, vis, rngs = step_vmapped(
                states, alive, clp, aseq, vis, temp_sched, rngs, jnp.int32(t))
            term_any = np.asarray(terminated.any(axis=1))
            alive_any = np.asarray(alive.any(axis=1))
            for w in range(W):
                if done[w]:
                    continue
                if term_any[w]:
                    bidx = int(np.asarray(terminated[w]).argmax())
                    sl = t + 1
                    solved[w] = True; solved_len[w] = sl
                    packed[w] = np.asarray(aseq[w, bidx, :sl]).tolist()
                    done[w] = True
                elif not alive_any[w]:
                    done[w] = True
            if all(done):
                break
        out = []
        for w, (idx, seed, temp, temp_end) in enumerate(elements):
            out.append({"idx": int(idx), "seed": int(seed),
                        "solved": bool(solved[w]),
                        "path_length": int(solved_len[w]) if solved[w] else -1,
                        "packed_path": packed[w] if solved[w] else []})
        return out

    def run_batch(elements, wave_size, t_cap=None):
        results = []
        i = 0
        while i < len(elements):
            chunk = elements[i:i + wave_size]
            pad = wave_size - len(chunk)
            padded = chunk + [chunk[-1]] * pad if pad else chunk
            wave_out = run_wave(padded, t_cap=t_cap)
            results.extend(wave_out[:len(chunk)])   # drop padded duplicates
            i += wave_size
        return results

    return run_wave, run_batch

RUN_WAVE, RUN_BATCH = make_engine(B)
print("beam engine built for B =", B)

## 10. Correctness guard + beam positive controls  *(`[unverified]` -- GPU/Colab path, not executed locally)*

In [ ]:
# Correctness guard + beam positive controls (run on the 2 control indices 16,17).
# (a) batched engine must reproduce released beam_search.py byte-for-byte; (b) beam at B must
# actually SOLVE the controls -- so a later "none of the 16 solved" can't be a broken harness.
if not CONFIG.get("RUN_GUARD", True):
    print("RUN_GUARD=False -> guard skipped (already verified). Beam run starts on a clean GPU.")
else:
    import pandas as pd, ast as _ast
    CTRL = CONTROL_IDX                                   # [16, 17] from ## 5
    guard_csv = str(RUN_DIR / "_guard.csv")
    cmd = [sys.executable, "beam/beam_search.py",
           "--ckpt_path", CONFIG["CKPT"], "--dataset", CONFIG["DATASET_STEM"],
           "--training_dataset", CONFIG["TRAINING_DATASET"],
           "--start", str(CTRL[0]), "--end", str(CTRL[-1] + 1),
           "--beam_width", str(B), "--max_steps", str(T),
           "--alpha", "0.0", "--temperature", "0.0", "--temp_end", "0.0",
           "--seed", "0", "--out_csv", guard_csv]
    print("oracle:", " ".join(cmd)); subprocess.run(cmd, check=True)
    ref = pd.read_csv(guard_csv)
    def _pp(p): return _ast.literal_eval(p) if isinstance(p, str) and p.strip().startswith("[") else []
    ref_map = {int(r.presentation_idx): (bool(r.solved), int(r.path_length), _pp(r.path))
               for r in ref.itertuples()}
    elements = [(i, 0, 0.0, 0.0) for i in CTRL]
    for ws in (1, len(CTRL)):
        got = RUN_BATCH(elements, ws)
        for g in got:
            rs, rl, rp = ref_map[g["idx"]]
            assert g["solved"] == rs and g["path_length"] == rl and g["packed_path"] == rp, (
                f"GUARD FAIL idx {g['idx']} ws={ws}: ours {(g['solved'], g['path_length'])} vs {(rs, rl)}")
        print(f"guard ws={ws}: OK ({len(got)} match released beam_search.py)")
    # positive control: beam at B MUST solve both controls under production config
    for g in RUN_BATCH(elements, len(CTRL)):
        assert g["solved"], f"BEAM POSITIVE CONTROL FAILED: idx {g['idx']} unsolved at B={B}"
    print("GUARD + beam positive controls PASSED (engine matches oracle AND solves the controls).")
    print(">>> Next: set CONFIG['RUN_GUARD']=False, Runtime->Restart session, re-run for the heavy "
          "beam stress pass (frees the guard's retained GPU memory). <<<")

## 11. Beam stress run (B=32,768 x 5 seeds, horizon T=150, resumable)  *(`[unverified]` -- GPU/Colab path, not executed locally)*

In [ ]:
# Beam stress run -- B x 5 seeds over the 16 targets (indices 0..15), resumable, full horizon T.
TARGET_IDX = list(range(16))                         # 0..15 targets; 16,17 are controls (guard only)

# wave size: HARD cap min(auto_wave, WAVE_CAP). Peak ~ a single fused actor-head alloc scaling with
# W*B (NOT linearly): W=3*32768 = 98,304 == the proven-OK W=6*16384 point (~59 GB on 80 GB);
# W=4*32768 = 131,072 > the W=7*16384 OOM point. So the cap can never drift into an OOM.
def _gb(key):
    try: return jax.devices()[0].memory_stats().get(key, 0) / 1e9
    except Exception: return 0.0
FIXED_GB, PER_PRES_GB_AT_16384 = 1.5, 9.65
def auto_wave(Bw):
    budget = 0.75 * (_gb("bytes_limit") or 80.0)
    return max(1, int((budget - FIXED_GB) // (PER_PRES_GB_AT_16384 * (Bw / 16384.0))))
wave_size = min(auto_wave(B), CONFIG["WAVE_CAP"])
print(f"wave_size = {wave_size}  (auto={auto_wave(B)}, hard cap WAVE_CAP={CONFIG['WAVE_CAP']}; peak ~ W*B)")

def _temp_for(seed):
    if seed == 0:
        return (0.0, 0.0)                                              # deterministic primary pass
    if seed == CONFIG["SUSTAINED_SEED"]:
        return (CONFIG["SUSTAINED_TEMP"], CONFIG["SUSTAINED_TEMP"])    # FA B5: one sustained-high-temp seed
    return (CONFIG["TEMP"], CONFIG["TEMP_END"])                        # linear Gumbel explore 1.0->0.0

done = load_done_ids()
elements = []                                          # (idx, seed, temp, temp_end)
for i in TARGET_IDX:
    key = TARGETS[i]["key"]
    for seed in CONFIG["SEEDS"]:
        if ("beam", key, seed) in done:
            continue
        tlo, thi = _temp_for(seed)
        elements.append((i, seed, tlo, thi))
print(f"beam todo: {len(elements)} runs (16 targets x {len(CONFIG['SEEDS'])} seeds; "
      f"skipped {sum(1 for x in done if x[0]=='beam')} done)")

for wi in range(0, len(elements), wave_size):
    chunk = elements[wi:wi + wave_size]
    pad = wave_size - len(chunk)
    padded = chunk + [chunk[-1]] * pad if pad else chunk
    w0 = time.time()
    out = RUN_WAVE(padded, t_cap=None)                 # full search horizon T (no cap)
    dt = time.time() - w0
    for g in out[:len(chunk)]:
        t = TARGETS[g["idx"]]
        append_record({"rec_type": "beam", "key": t["key"], "name": t["name"], "group": t["group"],
                       "seed": g["seed"], "beam_solved": g["solved"], "beam_len": g["path_length"],
                       "packed_path": g["packed_path"], "beam_width": B, "horizon": T,
                       "wall_time": round(dt / len(chunk), 2)})
    n_solved = sum(g["solved"] for g in out[:len(chunk)])
    print(f"wave {wi // wave_size}: {len(chunk)} runs in {dt:.0f}s "
          f"({'warmup+' if wi == 0 else ''}solved {n_solved})")
print("beam stress pass complete.")

## 12. Validate every beam solve by replay  *(`[unverified]` -- GPU/Colab path, not executed locally)*

In [ ]:
# Validate every beam solve by replaying its packed path in a fresh env (drop any that fail).
recs = load_records()
beam_solved = [r for r in recs if r["rec_type"] == "beam" and r["beam_solved"]]
maxpl = max((r["beam_len"] for r in beam_solved), default=1)
val_env = ACS(n_gen=2, max_length=L, max_steps_in_episode=max(maxpl, 1),
              is_reward_sparse=False, initial_states_file=CONFIG["DATASET_STEM"])
key2idx = {t["key"]: i for i, t in enumerate(TARGETS)}      # targets are dataset rows 0..15
bad = []
for r in beam_solved:
    idx = key2idx[r["key"]]
    term, nsteps, _ = replay_packed_path(val_env, idx, r["packed_path"], max_length=L)
    if not (term and nsteps == r["beam_len"]):
        bad.append((r["name"], r["seed"], bool(term), int(nsteps), r["beam_len"]))
print(f"replay: {len(beam_solved)} beam solves checked, {len(bad)} failed")
for b in bad:
    print("  FAIL name/seed/term/nsteps/claimed:", b)
assert not bad, "some beam solves failed replay validation -- not trusted"
print("OK: every beam solve reaches trivial in its recorded length." if beam_solved
      else "no beam solves to validate (expected for these hard targets).")

## 13. Verdict + relabel flags (named->HALT, cousin->three-way)

In [ ]:
# Verdict + relabel flags. Provenance-separated distances (greedy_len is NOT an env d-o-t -- FA B2).
# Named-anchor "solve" -> HALT/escalate (bug or AC refutation); only cousins take the relabel track.
import shutil
recs = load_records()
g_by = {r["key"]: r for r in recs if r["rec_type"] == "greedy"}
b_by = {}
for r in recs:
    if r["rec_type"] == "beam":
        b_by.setdefault(r["key"], []).append(r)

LEN12_PREDICTED_TRIV = "YXXYx|YYYYXyX"        # length-12 cousin: theory says trivializable (FA B1/a)
summary, halts, relabels = [], [], []
for t in TARGETS:
    k = t["key"]; g = g_by.get(k); bs = b_by.get(k, [])
    beam_solved_recs = [r for r in bs if r["beam_solved"]]
    beam_len_min = min((r["beam_len"] for r in beam_solved_recs), default=None)
    greedy_solved = bool(g and g["greedy_solved"])
    relabel_dot = beam_len_min                                  # ONLY the env-validated beam path
    ak_touched = g["ak_key_touched"] if g else None
    trivialized = bool(beam_solved_recs) or greedy_solved
    if t["group"] == "named":
        status = "ANOMALY_SOLVED" if trivialized else "survived_strongest_search"
        if trivialized:
            halts.append(t["name"])
    else:                                                       # short_hard cousin -> three-way
        if trivialized:
            status = "trivialized"
            prov = "beam(env-validated)" if beam_len_min is not None else "greedy(needs env-packed reconstruction)"
            relabels.append((t["name"], relabel_dot, prov))
        elif ak_touched:
            status = "certified_AK_equivalent"
        else:
            status = "still_unsolved"
    summary.append({"key": k, "name": t["name"], "group": t["group"], "total_len": t["total_len"],
                    "status": status, "greedy_solved": greedy_solved,
                    "greedy_len": (g["greedy_len"] if g else None), "beam_len_min": beam_len_min,
                    "relabel_dot": relabel_dot, "best_len_reached": (g["best_len_reached"] if g else None),
                    "ak_key_touched": ak_touched, "greedy_nodes": (g["nodes_visited"] if g else None),
                    "greedy_stop": (g["stop_reason"] if g else None),
                    "beam_width": B, "horizon": T, "seeds": CONFIG["SEEDS"]})

out = {"verdict_note": ("'survived the strongest available search at budget X' -- NOT 'proven hard'. "
                        "Neither engine is a neutral hardness oracle (greedy = length-greedy GBFS over a "
                        "global visited set; beam = the easy-hump-trained 610model policy, OOD on these)."),
       "config": {kk: CONFIG[kk] for kk in ("GREEDY_NODES", "B", "T", "SEEDS")},
       "targets": summary}
with open(SUMMARY_PATH, "w") as f:
    json.dump(out, f, indent=2)
os.makedirs(os.path.dirname(CONFIG["REPO_SUMMARY"]) or ".", exist_ok=True)
with open(CONFIG["REPO_SUMMARY"], "w") as f:
    json.dump(out, f, indent=2)
if RESULTS_JSONL.exists():
    os.makedirs(os.path.dirname(CONFIG["REPO_RESULTS"]) or ".", exist_ok=True)
    shutil.copyfile(RESULTS_JSONL, CONFIG["REPO_RESULTS"])      # repo mirror for committing

print("=" * 76)
for r in summary:
    flag = ""
    if r["group"] == "short_hard" and r["key"] == LEN12_PREDICTED_TRIV and r["status"] != "trivialized":
        flag = "   <-- len-12, PREDICTED-TRIVIALIZABLE but unsolved: investigate harness, not 'hard'"
    print(f"[{r['group']:9s}] {r['name']:18s} {r['status']:25s} "
          f"best_len={r['best_len_reached']} AK={r['ak_key_touched']} relabel_dot={r['relabel_dot']}{flag}")
print("=" * 76)
if halts:
    print("\n*** HALT / ESCALATE -- named anchor(s) reported SOLVED:", halts)
    print("    A replay-validated AK(n) solve is a refutation of Andrews-Curtis (extraordinary) OR --")
    print("    far more likely -- an env/canon/hash-dedup bug. DO NOT relabel; dump the move sequence")
    print("    + dedup hashes and investigate before trusting.")
if relabels:
    print("\n*** RELABEL-NEEDED -- cousin(s) trivialized (separate reviewed step; archive NOT edited):")
    for nm, dot, prov in relabels:
        print(f"    {nm}: finite d-o-t = {dot}  (provenance = {prov})")
if not halts and not relabels:
    print("\nAll 16 targets survived the strongest available search; high/censored labels confirmed "
          "under this budget (NOT 'proven hard').")
print("\nsummary written ->", SUMMARY_PATH)
print("repo mirror     ->", CONFIG["REPO_SUMMARY"], "+", CONFIG["REPO_RESULTS"])

## 14. Goal-specific sanity checks  *((d) is GPU-only, `[unverified]`)*

In [ ]:
# Goal-specific sanity checks (Field Advisor d). (a)/(b)/(c) hard-assert; (d) is best-effort.
import numpy as np

# (a) env invariants
assert L == 24, "L (action packing) is the invariant; T (search horizon) is a tunable knob"
assert isinstance(T, int) and T > 0, "search horizon T must be a positive int"
print(f"(a) env config OK: L=24, search horizon T={T} (not the training NUM_STEPS=96).")

# (b) the 8 cousins must still be `censored` in the LIVE dot_archive (guard against archive drift)
arch = {}
with open(CONFIG["DOT_ARCHIVE"]) as f:
    for line in f:
        d = json.loads(line)
        arch[canon.canon_key(d["r1"], d["r2"])[0]] = d
drift = [t["name"] for t in TARGETS if t["group"] == "short_hard"
         and not (arch.get(t["key"]) and arch[t["key"]].get("censored", False))]
print(f"(b) live-archive cousins still censored: {'OK' if not drift else 'DRIFT -> ' + str(drift)}")

# (c) distance-as-class-invariant: solve a control from a 2nd canonical representative -> same length
c = CONTROLS[0]
alt_r2 = c["r2"][1:] + c["r2"][0]                       # a rotation = same class, different representative
l1 = (lambda p: len(p) - 1 if p else -1)(
    ACRelatorSolver(c["r1"], c["r2"], max_nodes=300000, stop_early=False, verbose=False).solve()[0])
l2 = (lambda p: len(p) - 1 if p else -1)(
    ACRelatorSolver(c["r1"], alt_r2, max_nodes=300000, stop_early=False, verbose=False).solve()[0])
print(f"(c) distance-invariant on {c['name']}: rep1 len={l1}, rotated-rep len={l2} -> "
      f"{'OK (equal)' if l1 == l2 and l1 > 0 else 'DIFFER -- investigate'}")

# (d) action-space equivalence (best-effort): greedy get_neighbors_nj vs env (i,j,k1,k2) S-moves
#     on a sample state. Expect env-neighbor canonical keys subset of greedy's (env gate is stricter:
#     it no-ops on PRE-cyclic_reduce length>24, greedy on POST-reduce length<=24).
try:
    t = TARGETS[8]                                      # a cousin, mid-size
    r1, r2 = t["r1"], t["r2"]
    self_k = canon.canon_key(r1, r2)[0]
    # greedy neighbor keys (post-reduce, per-relator <=24)
    gk = set()
    for nr1, nr2 in get_neighbors_nj(str_to_arr(r1), str_to_arr(r2)):
        a, b = reduce_relator_nj(nr1), reduce_relator_nj(nr2)
        if len(a) <= 24 and len(b) <= 24:
            c1, c2 = canonical_pair_nj(a, b)
            gk.add(arr_to_str(c1) + "|" + arr_to_str(c2))
    # env neighbor keys (step every packed action from this exact state)
    k0 = jax.random.PRNGKey(0)
    _, st0 = env.reset_env(k0, env_params, idx=jnp.int32(0), sample=jnp.bool_(False), probs=None)
    lit = jnp.asarray(canon.strs_to_presentation_literal(r1, r2, 24), dtype=st0.x.dtype)
    st = st0.replace(x=lit)
    acts = jnp.stack([jnp.asarray(decode_action(a, 24), dtype=jnp.int32) for a in range(n_actions)])
    keys = jax.random.split(k0, n_actions)
    _, ns, _, _, _ = jax.vmap(env.step_env, in_axes=(0, None, 0, None))(keys, st, acts, env_params)
    ek = set()
    for x in np.asarray(ns.x):
        rr1, rr2 = canon.env_state_to_strs(x, 24)
        kk = canon.canon_key(rr1, rr2)[0]
        if kk != self_k:                                # drop no-ops (state unchanged)
            ek.add(kk)
    missing = ek - gk
    print(f"(d) action-space on {t['name']}: |greedy|={len(gk)} |env|={len(ek)} "
          f"env_not_in_greedy={len(missing)} -> {'OK (env subset of greedy)' if not missing else 'MISMATCH'}")
except Exception as e:
    print(f"(d) action-space check skipped ({type(e).__name__}: {e})")

print("sanity checks complete.")